# Brownian Motion / Wiener Process

**Concept 32** of the math-foundations learning map.

This notebook simulates standard Brownian motion $W_t$ on $[0,1]$, visualises sample paths, and empirically verifies two of its defining properties:

1. The marginal $W_t \sim \mathcal{N}(0, t)$, so $\mathrm{Var}(W_t) = t$.
2. The covariance kernel $\mathbb{E}[W_s W_t] = \min(s, t)$.

Everything below uses only the Python standard library (plus matplotlib for plotting).


## 1. Setup and Box-Muller sampler

We need standard-normal increments $Z_k \sim \mathcal{N}(0,1)$ without `numpy.random`. Box-Muller converts two uniforms on $(0,1)$ into one standard normal:
$$Z = \sqrt{-2 \log U_1} \cos(2\pi U_2).$$


In [ ]:
import math
import random

def box_muller(rng):
    u1 = rng.random()
    while u1 == 0.0:
        u1 = rng.random()
    u2 = rng.random()
    return math.sqrt(-2.0 * math.log(u1)) * math.cos(2.0 * math.pi * u2)

rng = random.Random(20260514)
print("3 N(0,1) samples:", [round(box_muller(rng), 4) for _ in range(3)])


## 2. Simulate Brownian motion on $[0,1]$

We discretise $[0,1]$ into $N = 1000$ equal steps of size $\Delta t = 1/N$. The recursion
$$W_{(k+1)/N} = W_{k/N} + \sqrt{\Delta t}\, Z_k$$
generates one sample path. By construction each increment is $\mathcal{N}(0, \Delta t)$ and increments over disjoint intervals are independent.


In [ ]:
def simulate_path(N, rng):
    dt = 1.0 / N
    sqrt_dt = math.sqrt(dt)
    W = [0.0] * (N + 1)
    for k in range(N):
        W[k + 1] = W[k] + sqrt_dt * box_muller(rng)
    return W

N = 1000
paths = [simulate_path(N, rng) for _ in range(5)]
half = N // 2
print("Values at t = 0.5 across 5 sample paths:")
for i, p in enumerate(paths, 1):
    print(f"  path {i}: W_0.5 = {p[half]:+.4f}")


## 3. Plot the sample paths

The paths are continuous (we connect successive points with line segments), but visibly rough — a hallmark of Brownian motion's non-differentiability.


In [ ]:
import matplotlib.pyplot as plt

ts = [k / N for k in range(N + 1)]
plt.figure(figsize=(9, 4))
for i, p in enumerate(paths):
    plt.plot(ts, p, lw=0.9, label=f"path {i+1}")
plt.axhline(0, color='k', lw=0.5)
plt.title("Five sample paths of standard Brownian motion on [0,1]")
plt.xlabel("t")
plt.ylabel("W_t")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 4. Verify $\mathrm{Var}(W_{0.5}) \approx 0.5$

By definition, $W_{0.5} \sim \mathcal{N}(0, 0.5)$. We estimate its variance from 1000 independent paths and expect the answer to be near $0.5$.


In [ ]:
n_paths = 1000
samples_half = []
for _ in range(n_paths):
    w = 0.0
    sqrt_dt = math.sqrt(1.0 / N)
    for _ in range(half):
        w += sqrt_dt * box_muller(rng)
    samples_half.append(w)

mean = sum(samples_half) / n_paths
var = sum((x - mean) ** 2 for x in samples_half) / (n_paths - 1)
print(f"  empirical mean     = {mean:+.4f}   (expected 0)")
print(f"  empirical variance = {var:.4f}    (expected 0.5)")
print(f"  relative error     = {abs(var - 0.5) / 0.5 * 100:.2f} %")


## 5. Histogram of $W_{0.5}$ versus the theoretical density

If the simulation is faithful, the empirical histogram should match the $\mathcal{N}(0, 0.5)$ density.


In [ ]:
import matplotlib.pyplot as plt
import math

plt.figure(figsize=(8, 4))
plt.hist(samples_half, bins=40, density=True, alpha=0.6, label='empirical')

# Overlay N(0, 0.5) density
xs = [-3 + 0.05 * k for k in range(121)]
sigma2 = 0.5
ys = [math.exp(-x * x / (2 * sigma2)) / math.sqrt(2 * math.pi * sigma2) for x in xs]
plt.plot(xs, ys, 'r-', lw=2, label='N(0, 0.5) density')
plt.title("Distribution of W_{0.5} over 1000 paths")
plt.xlabel("W_{0.5}")
plt.ylabel("density")
plt.legend()
plt.tight_layout()
plt.show()


## 6. Empirical check of the covariance kernel

Theorem: $\mathbb{E}[W_s W_t] = \min(s, t)$. We check this at $(s, t) = (0.3, 0.7)$ — expected value $0.3$.


In [ ]:
s_idx, t_idx = int(0.3 * N), int(0.7 * N)
prods = []
for _ in range(500):
    p = simulate_path(N, rng)
    prods.append(p[s_idx] * p[t_idx])

emp_cov = sum(prods) / len(prods)
print(f"  empirical E[W_0.3 * W_0.7] = {emp_cov:.4f}  (expected min(0.3, 0.7) = 0.30)")


## 7. Takeaways

- $W_t$ is built by accumulating independent Gaussian increments of variance $\Delta t$.
- The marginal at time $t$ is $\mathcal{N}(0, t)$ — confirmed empirically above.
- The covariance kernel $\min(s, t)$ — confirmed empirically above.
- These two properties, plus continuity of paths, fully characterise the law of Brownian motion. They underpin the SDEs that drive diffusion models such as DDPM and the score-based generative framework of Song et al. (2021).
